## TL;DR — Plain English

**What this notebook does:**
Builds the complete AF3-style training loop from scratch — every loss component, schedule, and memory trick.

**By the end, you will be able to:**
- Implement FAPE loss and explain why it's better than RMSD for training
- Combine 5 loss terms the way AF3 does (with correct weights)
- Write a training loop with gradient accumulation, warmup/cosine LR, gradient clipping
- Explain 3 memory optimization techniques: gradient checkpointing, chunked attention, BF16

**Why this matters for interviews:** "Implement a protein structure training loop" is a common
coding screen at computational biology ML teams and structural biology research labs. This notebook gives you the complete answer.


# Module 07/05 — AF3-Style Training Loop
## From Loss Functions to Full Training Infrastructure

**What you will build:** A complete training loop matching the AF3 paper:
- FAPE loss (backbone + all-atom)
- Distogram cross-entropy loss
- Masked MSA loss
- Violation loss (bond lengths / clashes)
- pLDDT prediction loss
- Gradient accumulation, warmup + cosine LR schedule
- Checkpointing with eval metrics

**Prerequisites:** Modules 07/01, 07/02, 07/03 — you should know what each loss term measures.

**Time:** ~2 hours of active study


## 🟢 Complete Beginner's Guide

**Assumed background:** Knows Python well, but has not implemented or studied a deep learning training loop.

### What you need to know before starting

- **Loss function** — a single number measuring how wrong the model's predictions are. Lower is better. The training loop's goal is to minimize this number.
- **Gradient** — the derivative of the loss with respect to each model parameter. It tells you: 'if I increase this weight by a tiny amount, does the loss go up or down, and by how much?'
- **Backpropagation** — the algorithm that computes all gradients efficiently by applying the chain rule through every layer.
- **Optimizer** — the algorithm that uses gradients to update weights (e.g. Adam: adjusts each weight by its gradient, scaled by past gradient history).
- **Batch** — a small subset of your training data processed together. Typical batch sizes: 8, 16, 32, 64 examples at once.
- **Epoch** — one full pass through the entire training dataset.
- **Learning rate** — how big a step the optimizer takes. Too large = overshoots; too small = never converges.

### The 5-line training loop skeleton (memorize this)

```python
for batch in dataloader:          # 1. Get a batch
    optimizer.zero_grad()         # 2. Clear old gradients
    loss = model(batch)           # 3. Forward pass (compute loss)
    loss.backward()               # 4. Backward pass (compute gradients)
    optimizer.step()              # 5. Update weights
```

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `DataLoader` | PyTorch utility that batches and shuffles your dataset |
| `model.train()` | Switches model to training mode (enables dropout, batch norm updates) |
| `model.eval()` | Switches model to evaluation mode (disables dropout) |
| `torch.no_grad()` | Context manager that disables gradient computation (saves memory) |
| `checkpoint` | A saved copy of model weights at a specific training step |
| `gradient clipping` | Capping gradient magnitude to prevent exploding gradients |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — map the AF3-specific training loop to the 5-line skeleton above.
2. **Run code cells one at a time** — print the loss value at each step and watch it decrease.
3. **Modify one number and re-run** — double the learning rate and observe what happens to the loss curve.

### If you're stuck

- **YouTube:** 'PyTorch Training Loop' by Andrej Karpathy — 'Neural Networks: Zero to Hero' (https://www.youtube.com/watch?v=VMj-3S1tku0)
- **YouTube:** 'Deep Learning Fundamentals' by Weights & Biases (https://www.youtube.com/playlist?list=PLD80i8An1OEGajeVo15ohAQYF1Ttle0lk)
- **Book:** *Deep Learning* by Goodfellow et al., Chapter 8 (Optimization for Training Deep Models).
- **Web:** https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html


---
## 1. Loss Architecture Overview

AF3 uses a **multi-task loss** where each head predicts something different:

```
Total loss = w_FAPE * L_FAPE
           + w_dist * L_distogram
           + w_msa  * L_masked_msa
           + w_viol * L_violations
           + w_lddt * L_plddt
```

Each weight is a hyperparameter. AF3 paper uses:
- L_FAPE: primary structure loss (backbone FAPE + all-atom FAPE)
- L_distogram: auxiliary — helps gradients flow early in training
- L_masked_msa: auxiliary — forces useful MSA representations
- L_violations: hard physics constraint (no bond length explosion, no steric clash)
- L_plddt: self-supervised confidence (predict your own lDDT score)

### WHY multiple losses?
Single FAPE loss has sparse gradients early — most positions are far from correct.
Distogram loss provides a denser training signal (N² predictions vs N residues).
The model learns geometry before fine details.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("AF3 Training Loss Components")
print("=" * 55)
print()
print("We implement each loss from scratch with numerical examples.")
print("Batch size B=2, sequence length L=30, for clarity.")
print()

B, L = 2, 30

---
## 2. FAPE Loss — Frame Aligned Point Error

### The Core Idea

Standard RMSD measures global fit. FAPE measures **local fit** by comparing coordinates in each residue's local reference frame.

**Why local?** A protein can be globally wrong (wrong fold) but locally right (correct local geometry). Or globally right but with one bad loop. FAPE distinguishes these cases — RMSD doesn't.

### Math

For each frame $i$, transform all $j$ atoms into frame $i$'s coordinate system:
$$\text{FAPE} = \frac{1}{N_\text{frames}} \sum_i \frac{1}{N_\text{atoms}} \sum_j \min\left(d_{ij}, d_\text{clamp}\right)$$

where $d_{ij} = \|\hat{R}_i^T(\hat{x}_j - \hat{t}_i) - R_i^T(x_j - t_i)\|$

$d_\text{clamp}=10\text{Å}$ prevents distant errors from dominating.


In [ ]:
# ─── FAPE Loss Implementation ───────────────────────────────

def fape_loss(pred_coords, true_coords, pred_frames_R, pred_frames_t,
              true_frames_R, true_frames_t, d_clamp=10.0, eps=1e-8):
    # pred_coords: (B, L, 3) — predicted CA positions
    # true_coords: (B, L, 3) — true CA positions
    # pred_frames_R: (B, L, 3, 3) — predicted rotation per residue
    # pred_frames_t: (B, L, 3) — predicted translation per residue
    # true_frames_R: (B, L, 3, 3) — true rotation per residue
    # true_frames_t: (B, L, 3) — true translation per residue

    B, L, _ = pred_coords.shape

    # For each frame i, transform all atoms j
    # pred local:  R_i^T @ (x_j - t_i)
    # true local:  R_i^T @ (x_j - t_i)

    # Expand: (B, L_frames, 1, 3) and (B, 1, L_atoms, 3)
    pred_atoms = pred_coords.unsqueeze(1)   # (B, 1, L, 3)
    true_atoms = true_coords.unsqueeze(1)   # (B, 1, L, 3)

    pred_t_exp = pred_frames_t.unsqueeze(2) # (B, L, 1, 3)
    true_t_exp = true_frames_t.unsqueeze(2) # (B, L, 1, 3)

    # Translate to frame origin
    pred_local = pred_atoms - pred_t_exp   # (B, L_frames, L_atoms, 3)
    true_local = true_atoms - true_t_exp   # (B, L_frames, L_atoms, 3)

    # Rotate into frame
    # R^T @ v = einsum('b f i j, b f a j -> b f a i', R, v)
    pred_R_T = pred_frames_R.transpose(-1, -2)  # (B, L, 3, 3)
    true_R_T = true_frames_R.transpose(-1, -2)

    # (B, L_frames, L_atoms, 3)
    pred_in_frame = torch.einsum('bfij,bfaj->bfai', pred_R_T, pred_local)
    true_in_frame = torch.einsum('bfij,bfaj->bfai', true_R_T, true_local)

    # Distance between predicted and true local coordinates
    diff = pred_in_frame - true_in_frame  # (B, L_frames, L_atoms, 3)
    dist = torch.sqrt((diff ** 2).sum(-1) + eps)  # (B, L_frames, L_atoms)

    # Clamp large errors
    dist_clamped = torch.clamp(dist, max=d_clamp)

    # Average over atoms then frames
    loss = dist_clamped.mean(dim=-1).mean(dim=-1)  # (B,)
    return loss.mean()


# Test FAPE with known values
def random_rotation(B, L):
    # Generate random rotation matrices via QR decomposition
    M = torch.randn(B, L, 3, 3)
    Q, _ = torch.linalg.qr(M)
    # Ensure proper rotation (det=+1)
    det = torch.linalg.det(Q)
    Q = Q * det.unsqueeze(-1).unsqueeze(-1)
    return Q

B, L = 2, 30
pred_coords = torch.randn(B, L, 3) * 5
true_coords = torch.randn(B, L, 3) * 5

pred_R = random_rotation(B, L)
true_R = random_rotation(B, L)
pred_t = torch.randn(B, L, 3)
true_t = torch.randn(B, L, 3)

# Perfect prediction: FAPE should be 0
fape_perfect = fape_loss(true_coords, true_coords, true_R, true_t, true_R, true_t)
fape_random = fape_loss(pred_coords, true_coords, pred_R, pred_t, true_R, true_t)

print(f"FAPE loss:")
print(f"  Perfect prediction: {fape_perfect:.6f}  (should be ~0)")
print(f"  Random prediction:  {fape_random:.4f} Angstroms")
print()
print("d_clamp=10A means errors > 10A contribute equally (avoids outlier dominance)")
print("AF3 uses separate FAPE for backbone frames vs all-atom frames")

---
## 3. Distogram Loss — Auxiliary Structural Signal

### Why Distogram?

Predicting the full distance matrix (N×N bins) gives dense gradients even when 3D structure is wrong.

Each pair $(i,j)$ predicts which of 64 distance bins the Cβ-Cβ distance falls into.
This is a cross-entropy classification task.

**Key insight**: A model can learn good distance predictions before learning correct 3D structure.
Then 3D structure emerges from the distance constraints.


In [ ]:
# ─── Distogram Loss Implementation ──────────────────────────

def make_distogram_targets(true_coords, n_bins=64, min_d=2.0, max_d=22.0):
    # true_coords: (B, L, 3) — CA or CB positions
    # Returns bin indices (B, L, L)
    dists = torch.cdist(true_coords, true_coords)   # (B, L, L)
    # Log-spaced bins (matches AF3)
    bins = torch.exp(torch.linspace(np.log(min_d), np.log(max_d), n_bins + 1))
    bin_targets = torch.bucketize(dists, bins[:-1].to(dists.device))
    bin_targets = bin_targets.clamp(0, n_bins - 1)
    return bin_targets  # (B, L, L)

def distogram_loss(pred_logits, true_coords, n_bins=64):
    # pred_logits: (B, L, L, n_bins) — predicted bin probabilities
    # true_coords: (B, L, 3)
    targets = make_distogram_targets(true_coords, n_bins=n_bins)  # (B, L, L)
    B, L, _, n_bins = pred_logits.shape
    loss = F.cross_entropy(
        pred_logits.reshape(B * L * L, n_bins),
        targets.reshape(B * L * L),
    )
    return loss

# Simulate distogram head output
B, L, n_bins = 2, 30, 64
true_coords = torch.randn(B, L, 3) * 5
pred_logits = torch.randn(B, L, L, n_bins)  # uninitialized head

loss_dist = distogram_loss(pred_logits, true_coords)
print(f"Distogram loss (random logits): {loss_dist:.4f}")
print(f"  Expected for random: log({n_bins}) = {np.log(n_bins):.4f}")

# Sanity: perfect logits → loss ≈ 0
targets = make_distogram_targets(true_coords, n_bins=n_bins)
perfect_logits = torch.zeros(B, L, L, n_bins)
perfect_logits.scatter_(-1, targets.unsqueeze(-1), 100.0)
loss_perfect = distogram_loss(perfect_logits, true_coords)
print(f"Distogram loss (perfect logits): {loss_perfect:.6f}  (→ 0)")

---
## 4. Masked MSA Loss — Self-Supervised Representation Learning

### Analogy to BERT

BERT masks 15% of input tokens and predicts them. AF3 does the same to the MSA:
- Mask 15% of MSA positions randomly
- Predict the original AA at masked positions
- Forces the model to learn evolutionary relationships, not just copy input

**Why this helps 3D structure:**
The MSA features encode evolutionary constraints (co-evolution). Forcing reconstruction
of masked positions requires understanding the co-evolution signal — exactly what
Pairformer needs for accurate structure prediction.


In [ ]:
# ─── Masked MSA Loss Implementation ─────────────────────────

def masked_msa_loss(pred_logits, true_tokens, mask):
    # pred_logits: (B, D, L, 23) — 23 = 20 AA + gap + mask token + unknown
    # true_tokens: (B, D, L) — true AA indices at masked positions
    # mask: (B, D, L) — 1 where position is masked
    B, D, L, vocab = pred_logits.shape
    loss = F.cross_entropy(
        pred_logits.reshape(B * D * L, vocab),
        true_tokens.reshape(B * D * L),
        reduction='none'
    ).reshape(B, D, L)
    # Only average over MASKED positions
    masked_loss = (loss * mask).sum() / (mask.sum() + 1e-8)
    return masked_loss

# Simulate
B, D, L, vocab = 2, 128, 30, 23  # D=128 MSA depth
mask_rate = 0.15

true_tokens = torch.randint(0, 20, (B, D, L))
pred_logits = torch.randn(B, D, L, vocab)

# Create random mask
mask = (torch.rand(B, D, L) < mask_rate).float()
n_masked = mask.sum().item()

loss_msa = masked_msa_loss(pred_logits, true_tokens, mask)
print(f"Masked MSA loss: {loss_msa:.4f}")
print(f"  Masked positions: {n_masked:.0f} / {B*D*L} ({100*mask_rate:.0f}%)")
print(f"  Expected random:  {np.log(vocab):.4f}")
print()
print("NOTE: AF3 uses 3 masking strategies:")
print("  1. Replace with [MASK] token (85% of masked positions)")
print("  2. Replace with random AA (10%)")
print("  3. Keep original (5%) — prevents model from ignoring unmasked positions")

---
## 5. Violation Loss — Physics Constraints

### What Can Go Wrong Without This?

Pure ML loss can produce chemically impossible structures:
- Bond lengths too long or short (N-CA ~1.46Å, CA-C ~1.52Å)
- Steric clashes: two heavy atoms < 1.5Å apart (van der Waals radius violation)

Violation loss penalizes these directly. It's not differentiable everywhere, so AF3 uses soft versions.

**Typical weights:** violation loss is small (0.02-0.1) — just a regularizer, not the primary objective.


In [ ]:
# ─── Violation Loss Implementation ──────────────────────────

def bond_length_violation(coords, ideal_length=3.8, tolerance=0.75):
    # Consecutive CA distances should be ~3.8 Angstroms (typical for protein chain)
    # coords: (B, L, 3)
    ca_dists = torch.norm(coords[:, 1:] - coords[:, :-1], dim=-1)  # (B, L-1)
    violation = F.relu(torch.abs(ca_dists - ideal_length) - tolerance)
    return violation.mean()

def steric_clash_loss(coords, clash_threshold=3.5):
    # Penalize CA-CA distances < clash_threshold for non-bonded pairs
    # coords: (B, L, 3)
    dists = torch.cdist(coords, coords)  # (B, L, L)
    # Create mask for non-adjacent pairs (ignore i,i and i,i+1)
    B, L, _ = coords.shape
    non_bonded = torch.ones(L, L, dtype=torch.bool)
    for k in range(-1, 2):
        non_bonded &= ~torch.eye(L, dtype=torch.bool).roll(k, 1)
    non_bonded_dists = dists[:, non_bonded]  # (B, n_pairs)
    clash = F.relu(clash_threshold - non_bonded_dists).mean()
    return clash

# Test on reasonable and unreasonable coords
B, L = 2, 30

# Good coords: linear chain with ~3.8Å CA-CA spacing
good_coords = torch.stack([
    torch.arange(L, dtype=torch.float) * 3.8,
    torch.zeros(L),
    torch.zeros(L)
], dim=1).unsqueeze(0).expand(B, -1, -1)

# Bad coords: random, may have clashes or wrong bond lengths
bad_coords = torch.randn(B, L, 3) * 3.0

viol_bond_good = bond_length_violation(good_coords)
viol_bond_bad = bond_length_violation(bad_coords)
viol_clash_good = steric_clash_loss(good_coords)
viol_clash_bad = steric_clash_loss(bad_coords)

print("Violation Loss Comparison:")
print(f"  {'Metric':<30} {'Good coords':>12} {'Random coords':>14}")
print(f"  {'Bond length violation':<30} {viol_bond_good.item():>12.4f} {viol_bond_bad.item():>14.4f}")
print(f"  {'Steric clash loss':<30} {viol_clash_good.item():>12.4f} {viol_clash_bad.item():>14.4f}")
print()
print("Good coords: sequential chain → near-zero bond violation, no clashes")
print("Random coords: arbitrary positions → large violations (chemically impossible)")

---
## 6. Complete Training Loop

Now combine everything into a full training loop with:
- Gradient accumulation (simulate large batch with small GPU memory)
- Linear warmup + cosine decay schedule
- Gradient clipping
- Checkpoint saving with best-metric tracking


In [ ]:
# ─── Minimal AF3-Style Model ──────────────────────────────────

class MiniAF3(nn.Module):
    # Tiny AF3-like model: Input → Pairformer blocks → structure + auxiliary heads
    def __init__(self, L=30, c_s=32, c_z=16, n_blocks=2, n_bins=64):
        super().__init__()
        self.L = L
        self.n_bins = n_bins

        # Input embedder
        self.embed_s = nn.Embedding(21, c_s)     # 20 AA + gap
        self.embed_z = nn.Linear(1, c_z)          # distance feature (recycling)

        # Pairformer blocks (simplified: 2 layers per block)
        blocks = []
        for _ in range(n_blocks):
            blocks.extend([
                nn.LayerNorm(c_z),
                nn.Linear(c_z, c_z),
            ])
        self.pair_blocks = nn.ModuleList([nn.Sequential(
            nn.LayerNorm(c_z), nn.Linear(c_z, c_z), nn.ReLU(), nn.Linear(c_z, c_z)
        ) for _ in range(n_blocks)])
        self.single_update = nn.ModuleList([nn.Sequential(
            nn.LayerNorm(c_s), nn.Linear(c_s, c_s * 2), nn.ReLU(), nn.Linear(c_s * 2, c_s)
        ) for _ in range(n_blocks)])

        # Structure head: predict CA coordinates (simplified: no diffusion)
        self.coord_head = nn.Sequential(
            nn.LayerNorm(c_s), nn.Linear(c_s, 64), nn.ReLU(), nn.Linear(64, 3)
        )

        # Distogram head
        self.distogram_head = nn.Sequential(
            nn.LayerNorm(c_z), nn.Linear(c_z, n_bins)
        )

        # pLDDT head
        self.plddt_head = nn.Sequential(
            nn.LayerNorm(c_s), nn.Linear(c_s, 50)   # 50 bins, 0.02 each
        )

    def forward(self, seq_tokens, dist_feat=None):
        # seq_tokens: (B, L) int
        B, L = seq_tokens.shape

        # Embed sequence
        s = self.embed_s(seq_tokens)   # (B, L, c_s)

        # Initialize pair repr from outer product
        z = (s.unsqueeze(2) + s.unsqueeze(1))  # (B, L, L, c_s)
        z = self.embed_z(z[..., :1])           # (B, L, L, c_z)

        # Pairformer blocks
        for pair_blk, single_blk in zip(self.pair_blocks, self.single_update):
            z = z + pair_blk(z)
            # Single update via mean of pair
            s = s + single_blk(s + z.mean(dim=2))

        # Heads
        coords = self.coord_head(s)           # (B, L, 3)
        dist_logits = self.distogram_head(z)  # (B, L, L, n_bins)
        plddt_logits = self.plddt_head(s)     # (B, L, 50)

        return coords, dist_logits, plddt_logits

# Quick check
model = MiniAF3(L=30, c_s=32, c_z=16, n_blocks=2)
seq = torch.randint(0, 20, (2, 30))
coords, dist_logits, plddt_logits = model(seq)
print(f"MiniAF3 output shapes:")
print(f"  coords:       {coords.shape}")
print(f"  dist_logits:  {dist_logits.shape}")
print(f"  plddt_logits: {plddt_logits.shape}")
print(f"  Parameters:   {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ─── Learning Rate Schedule ──────────────────────────────────

def af3_lr_schedule(step, warmup_steps=1000, total_steps=50000, base_lr=1e-3, min_lr=1e-5):
    # Linear warmup followed by cosine decay
    if step < warmup_steps:
        return base_lr * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    cosine_factor = 0.5 * (1 + np.cos(np.pi * progress))
    return min_lr + (base_lr - min_lr) * cosine_factor

# Visualize schedule
steps = np.arange(0, 50000, 100)
lrs = [af3_lr_schedule(s) for s in steps]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(steps, lrs)
axes[0].axvline(1000, color='orange', linestyle='--', label='Warmup end')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Learning rate')
axes[0].set_title('AF3-Style LR Schedule\n(warmup=1k, cosine decay to 50k)')
axes[0].legend()
axes[0].set_yscale('log')

# Show effect of different warmup lengths
for wup in [100, 500, 1000, 5000]:
    lrs_wup = [af3_lr_schedule(s, warmup_steps=wup) for s in steps]
    axes[1].plot(steps[:200], lrs_wup[:200], label=f'warmup={wup}')
axes[1].set_xlabel('Step (early training)')
axes[1].set_ylabel('Learning rate')
axes[1].set_title('Effect of Warmup Length\n(early steps)')
axes[1].legend()
plt.tight_layout()
plt.savefig('lr_schedule.png', dpi=100, bbox_inches='tight')
plt.show()
print("LR schedule saved")
print()
print("WHY warmup?")
print("  At step 0: random weights → large gradients → explosive updates")
print("  Warmup: start tiny lr → gradually increase → stable early training")
print("WHY cosine decay?")
print("  Smooth decrease → model 'settles' into minima at end of training")

In [ ]:
# ─── Full Training Loop ──────────────────────────────────────

def compute_total_loss(model, seq_tokens, true_coords,
                       w_fape=1.0, w_dist=0.3, w_viol=0.02):
    # Forward pass
    pred_coords, dist_logits, plddt_logits = model(seq_tokens)

    # FAPE loss (simplified: use identity frames)
    B, L = seq_tokens.shape
    I = torch.eye(3).unsqueeze(0).unsqueeze(0).expand(B, L, -1, -1)
    t = pred_coords.clone()
    t_true = true_coords.clone()
    loss_fape = fape_loss(pred_coords, true_coords, I, t, I, t_true)

    # Distogram loss
    loss_dist = distogram_loss(dist_logits, true_coords)

    # Violation loss
    loss_viol = bond_length_violation(pred_coords) + steric_clash_loss(pred_coords) * 0.1

    # Total
    total = w_fape * loss_fape + w_dist * loss_dist + w_viol * loss_viol

    return total, {
        'fape': loss_fape.item(),
        'dist': loss_dist.item(),
        'viol': loss_viol.item(),
        'total': total.item(),
    }

# ─── Training Configuration ──────────────────────────────────
model = MiniAF3(L=30, c_s=32, c_z=16, n_blocks=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0)  # LR set by scheduler
n_steps = 300
accumulate_steps = 4  # effective batch = 4 * batch_size
warmup_steps = 30
base_lr = 1e-3

history = {'fape': [], 'dist': [], 'viol': [], 'total': []}
running_losses = {k: 0.0 for k in history}

print(f"Training MiniAF3 for {n_steps} steps (gradient accumulation={accumulate_steps})")
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print()

optimizer.zero_grad()
for step in range(1, n_steps + 1):
    # Update learning rate
    lr = af3_lr_schedule(step, warmup_steps=warmup_steps,
                         total_steps=n_steps, base_lr=base_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    # Synthetic batch (in real training: load from data loader)
    B = 4
    seq = torch.randint(0, 20, (B, 30))
    # True coords: random protein-like (in real training: from PDB)
    true = torch.randn(B, 30, 3) * 5

    loss, losses = compute_total_loss(model, seq, true)
    loss = loss / accumulate_steps
    loss.backward()

    for k, v in losses.items():
        running_losses[k] += v / accumulate_steps

    # Step optimizer every accumulate_steps
    if step % accumulate_steps == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

        for k in history:
            history[k].append(running_losses[k] / accumulate_steps)
            running_losses[k] = 0.0

        if step % 50 == 0:
            print(f"  Step {step:4d} | lr={lr:.2e} | "
                  f"total={history['total'][-1]:.4f} | "
                  f"fape={history['fape'][-1]:.4f} | "
                  f"dist={history['dist'][-1]:.4f}")

# ─── Plot training curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
components = ['total', 'fape', 'dist', 'viol']
colors = ['black', 'steelblue', 'darkorange', 'firebrick']
for ax, comp, col in zip(axes, components, colors):
    ax.plot(history[comp], color=col, linewidth=2)
    ax.set_title(f'{comp.upper()} Loss')
    ax.set_xlabel('Optimizer step (per 4 grad accum steps)')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)

plt.suptitle('MiniAF3 Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("Training complete. Curves saved.")

In [ ]:
# ─── Checkpointing and Evaluation ────────────────────────────

import os

def save_checkpoint(model, optimizer, step, metrics, path):
    torch.save({
        'step': step,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'metrics': metrics,
    }, path)

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    return ckpt['step'], ckpt['metrics']

def compute_tm_score_approx(pred_coords, true_coords):
    # Simplified TM-score estimate (not the real algorithm — use TMscore binary for real use)
    # pred_coords: (L, 3), true_coords: (L, 3)
    L = pred_coords.shape[0]
    d0 = 1.24 * (L - 15) ** (1/3) - 1.8 if L > 21 else 0.5
    dists = torch.norm(pred_coords - true_coords, dim=-1)
    tm_terms = 1.0 / (1.0 + (dists / d0) ** 2)
    return tm_terms.mean().item()

# Simulate evaluation
model.eval()
with torch.no_grad():
    # Generate a test batch
    B_eval = 8
    seq_eval = torch.randint(0, 20, (B_eval, 30))
    true_eval = torch.randn(B_eval, 30, 3) * 5
    pred_eval, _, _ = model(seq_eval)

    # Per-structure metrics
    tm_scores = []
    rmsds = []
    for b in range(B_eval):
        tm = compute_tm_score_approx(pred_eval[b], true_eval[b])
        rmsd = torch.norm(pred_eval[b] - true_eval[b], dim=-1).mean().item()
        tm_scores.append(tm)
        rmsds.append(rmsd)

    print("Evaluation Results (random test, so low quality expected):")
    print(f"  Mean TM-score (approx): {np.mean(tm_scores):.4f}")
    print(f"  Mean RMSD:              {np.mean(rmsds):.4f} A")
    print()

# Save demo checkpoint
ckpt_path = '/tmp/mini_af3_demo.pt'
save_checkpoint(model, optimizer, step=n_steps,
                metrics={'tm_score': np.mean(tm_scores), 'rmsd': np.mean(rmsds)},
                path=ckpt_path)
print(f"Checkpoint saved: {ckpt_path}")
print(f"  Size: {os.path.getsize(ckpt_path) / 1024:.1f} KB")

# Reload and verify
model2 = MiniAF3(L=30, c_s=32, c_z=16, n_blocks=2)
opt2 = torch.optim.Adam(model2.parameters())
step_loaded, metrics_loaded = load_checkpoint(model2, opt2, ckpt_path)
print(f"  Loaded step: {step_loaded}, metrics: {metrics_loaded}")

---
## 7. Memory-Efficient Training Tricks

AF3 at full scale trains on structures up to 5120 tokens. These tricks are **essential** — without them, you run out of GPU memory.

### The Memory Problem

| Configuration | Memory for z pair tensor |
|---|---|
| L=100 | 100² × 128 × 4 bytes = 5 MB |
| L=500 | 500² × 128 × 4 bytes = 128 MB |
| L=2000 | 2000² × 128 × 4 bytes = 2 GB |
| L=5120 (AF3 max) | 5120² × 128 × 4 bytes = 13 GB |

And that's just ONE tensor. Triangle attention materializes (L,L,L) intermediates = L³ memory.

### Solutions

1. **Gradient checkpointing**: Don't store activations during forward pass — recompute them during backward. Saves ~60% memory at 33% compute cost.
2. **Chunked triangle attention**: Process triangle attention in chunks of size C along one dimension. Memory: O(N²C) instead of O(N³).
3. **Mixed precision (BF16)**: Use bfloat16 for forward pass (2x memory savings vs float32). AF3 prefers BF16 over FP16 due to wider exponent range.
4. **Sequence length bucketing**: Group similar-length sequences to minimize padding waste.


In [ ]:
# ─── Memory Tricks Demo ──────────────────────────────────────

print("Memory Efficiency Techniques for AF3-Scale Training")
print("=" * 60)

print()
print("1. GRADIENT CHECKPOINTING")
print("-" * 40)
print("   Standard: store ALL activations during forward → O(n_layers) memory")
print("   Checkpointed: store only checkpoints, recompute others on backward")
print()

import torch.utils.checkpoint as checkpoint

class CheckpointedBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x):
        return x + self.layers(x)

dim, L_test = 128, 100
x = torch.randn(L_test, dim, requires_grad=True)

blk = CheckpointedBlock(dim)

# Standard forward
y_std = blk(x)
mem_std = x.element_size() * x.nelement() + y_std.element_size() * y_std.nelement()

# Checkpointed forward (recomputes activations on backward)
y_ckpt = checkpoint.checkpoint(blk, x, use_reentrant=False)

print(f"   Input size: {x.shape} = {mem_std // 1024:.0f} KB")
print(f"   Output same? {torch.allclose(y_std, y_ckpt, atol=1e-5)}")
print(f"   In real training: saves ~60% activation memory for Pairformer blocks")

print()
print("2. CHUNKED ATTENTION (memory reduction for triangle attention)")
print("-" * 40)
print("   Full triangle attention: (L, L, L) intermediate = L^3 memory")
print("   Chunked: process C rows at a time → (C, L, L) intermediate")
print()

def chunked_row_softmax(z, chunk_size=32):
    # z: (L, L, d_k) — simulate triangle attention score computation
    L, _, d_k = z.shape
    out = torch.zeros_like(z)
    for start in range(0, L, chunk_size):
        end = min(start + chunk_size, L)
        chunk = z[start:end]  # (C, L, d_k)
        out[start:end] = torch.softmax(chunk, dim=1)
    return out

L_big = 200
z_test = torch.randn(L_big, L_big, 16)
out_chunked = chunked_row_softmax(z_test, chunk_size=32)
out_full = torch.softmax(z_test, dim=1)
print(f"   L={L_big} chunked result matches full: {torch.allclose(out_chunked, out_full, atol=1e-5)}")
print(f"   Memory ratio (C=32 vs full): {32}/{L_big} = {32/L_big:.2f}x reduction in one dimension")

print()
print("3. MIXED PRECISION (BF16)")
print("-" * 40)
from torch.cuda.amp import autocast
if torch.cuda.is_available():
    with autocast(dtype=torch.bfloat16):
        x_bf16 = torch.randn(100, 128, device='cuda', dtype=torch.bfloat16)
    print("   BF16 available on GPU")
else:
    x_fp32 = torch.randn(100, 128)
    x_bf16 = x_fp32.to(torch.bfloat16)
    x_fp16 = x_fp32.to(torch.float16)
    print(f"   FP32 bytes/element: {x_fp32.element_size()}")
    print(f"   BF16 bytes/element: {x_bf16.element_size()} (2x memory savings)")
    print(f"   FP16 bytes/element: {x_fp16.element_size()}")
    print(f"   BF16 vs FP16: same size, but BF16 has 8-bit exponent vs 5-bit")
    print(f"   → BF16 range: 1e-38 to 3e38 (same as FP32); FP16: 6e-5 to 65504")
    print(f"   → BF16 is safer for training: no overflow on large gradients")

---
## 8. Summary & What to Do Next

### What You Built
- FAPE loss with local frame alignment
- Distogram cross-entropy loss (auxiliary structural signal)
- Masked MSA loss (self-supervised sequence reconstruction)
- Violation loss (physics constraints)
- Complete training loop with gradient accumulation + warmup/cosine schedule
- Gradient checkpointing and chunked attention for memory efficiency

### What Real AF3 Training Adds
1. **Diffusion training**: the full forward + reverse diffusion process (not just coordinate regression)
2. **Template featurizer**: process template structures as additional input
3. **OpenFold weight loading**: initialize from pretrained weights
4. **Multi-GPU / TPU**: distributed training with DeepSpeed or JAX pmap
5. **Data pipeline**: asynchronous PDB/mmCIF loading, on-the-fly augmentation
6. **CASP evaluation**: GDT-TS, TM-score, lDDT computed via TM-align binary

### Study Path
- **Next:** Module 10/00 — Navigate OpenFold source code (see actual implementations)
- **Then:** Module 10/01 — Fine-tune on SKEMPI v2 for ΔΔG prediction

### Interview Readiness Checklist
- [ ] Can you explain why FAPE is better than RMSD for training? (local vs global)
- [ ] What does the distogram loss add that FAPE alone doesn't? (dense gradients early)
- [ ] Why gradient accumulation? (simulate larger batch on limited GPU memory)
- [ ] What does gradient checkpointing trade off? (memory for compute: recompute activations)
- [ ] Why BF16 over FP16 for AF3? (wider exponent range, safer for large gradients)


---
## 📚 Resources — Training AF3-Style Models

### University Courses on Training Deep Models
| Course | Key Lecture/Chapter |
|--------|---------------------|
| **[Stanford CS229 Lecture 4](https://cs229.stanford.edu/lectures-spring2022/main_notes.pdf)** | Gradient descent, stochastic vs mini-batch; free PDF |
| **[MIT 6.S191 Lab 1](https://introtodeeplearning.com/)** | Full training loop in PyTorch; free Colab |
| **[fast.ai Lesson 7-9](https://course.fast.ai/)** | Training tricks: 1-cycle LR, mixed precision, gradient clipping |
| **[Stanford CS224n Lecture 5](https://web.stanford.edu/class/cs224n/readings/)** | Optimization for NLP models; directly applicable |

### Essential Reading (Free)
- **[AF3 Methods Section 4](https://www.nature.com/articles/s41586-024-07487-w)** — exact training config (batch size 1, gradient accum 128, 50K steps, Adam)
- **[OpenFold utils/loss.py](https://github.com/aqlaboratory/openfold/blob/main/openfold/utils/loss.py)** — production FAPE, distogram, violations implementations
- **[SGDR: Cosine Annealing](https://arxiv.org/abs/1608.03983)** — the LR schedule used by AF3
- **[Lilian Weng: Learning Rate Schedules](https://lilianweng.github.io/posts/2022-01-02-train-optimizer/)** — complete survey of LR schedules

### AF3-Specific Training Facts (From the Paper)
```
Batch size:       1 chain per GPU step
Gradient accum:   128 steps → effective batch = 128
Learning rate:    1.8e-3 peak, warmup 1000 steps
Total steps:      50,000 (without finetuning)
Optimizer:        Adam (β1=0.9, β2=0.99, ε=1e-8)
Gradient clip:    0.1 max norm
Hardware:         128 TPU v4 cores
Training time:    ~2 weeks
```

### Debug Checklist for Training Loops (MIT 6.S191 Style)
1. **Overfit 1 sample first**: if loss doesn't go to ~0 on 1 example, your model or loss is broken
2. **Check loss scale**: cross-entropy on C classes should start near log(C)
3. **Gradients flow**: `for n,p in model.named_parameters(): print(n, p.grad.norm())` after 1 step
4. **Learning rate sensitivity**: train with lr × 10 and lr ÷ 10; correct lr should be in between
5. **Batch size effect**: larger batch → sharper minima; smaller → more noise but better generalization

### Interview Questions — Training Infrastructure

**Q1:** Why does AF3 use gradient accumulation with effective batch size 128 if it trains 1 sample per step?
> **A:** Large effective batch size stabilizes gradient estimates (less noise) and allows higher learning rates. GPUs are memory-limited, not compute-limited for single long-sequence proteins — so accumulation lets you simulate large batches without OOM errors.

**Q2:** What does gradient clipping (max_norm=0.1) prevent?
> **A:** Gradient explosion — particularly common in deep networks during early training when weights are random. Clipping ensures no single update step takes the model too far in one direction, preventing loss spikes.

**Q3:** BF16 vs FP16 — why does AF3 use BF16?
> **A:** BF16 has the same exponent range as FP32 (8 bits), just reduced mantissa precision (7 bits vs 23). FP16's 5-bit exponent causes overflow for values > 65504, which happens regularly in deep network training. BF16 avoids this while still saving 50% memory vs FP32.


## 📦 Real Datasets for AF3 Training Loop

### Dataset 1: CASP14 Targets
- **URL:** https://predictioncenter.org/casp14/targetlist.cgi
- **Direct download:** https://predictioncenter.org/download_area/CASP14/targets/
- **Format:** FASTA sequences + native PDB structures
- **Size:** ~50 MB total for all CASP14 targets
- **Why it matters:** CASP14 is the competition where AlphaFold2 first demonstrated near-experimental accuracy. These targets are the gold-standard benchmark.

### Dataset 2: PDB structure 7NRG (AF3 training example)
- **URL:** https://files.rcsb.org/download/7NRG.cif
- **Format:** mmCIF (modern PDB format)
- **Size:** ~500 KB
- **Why it matters:** 7NRG is a protein-ligand complex used in AF3 development. It demonstrates how AF3 handles small molecule inputs.

### Dataset 3: AF3 prediction JSON format example
- **Source:** AlphaFold Server output (https://alphafoldserver.com)
- **Format:** JSON with predicted coordinates, pLDDT, PAE matrix
- **Why it matters:** Understanding the output format is required for downstream analysis of AF3 predictions.

_Note: If downloads fail, the training loop in this notebook uses fully simulated protein data that demonstrates the same concepts._


In [ ]:
import requests
import os
import json

# ── Dataset 1: Download PDB 7NRG (mmCIF format) ─────────────────────────
PDB_ID = '7NRG'
MMCIF_URL = f'https://files.rcsb.org/download/{PDB_ID}.cif'
LOCAL_CIF = f'/tmp/{PDB_ID}.cif'

if not os.path.exists(LOCAL_CIF):
    print(f'Downloading {PDB_ID} from RCSB PDB...')
    try:
        r = requests.get(MMCIF_URL, timeout=60)
        r.raise_for_status()
        with open(LOCAL_CIF, 'w') as f:
            f.write(r.text)
        print(f'Saved to {LOCAL_CIF} ({os.path.getsize(LOCAL_CIF)//1024} KB)')
    except Exception as e:
        print(f'Download failed: {e}')
        print(f'Manual: wget -O {LOCAL_CIF} {MMCIF_URL}')
else:
    print(f'{LOCAL_CIF} already exists ({os.path.getsize(LOCAL_CIF)//1024} KB)')

# Parse with Biopython if available
try:
    from Bio.PDB import MMCIFParser
    if os.path.exists(LOCAL_CIF):
        parser = MMCIFParser(QUIET=True)
        structure = parser.get_structure(PDB_ID, LOCAL_CIF)
        chains = list(structure.get_chains())
        print(f'{PDB_ID} has {len(chains)} chain(s)')
        for chain in chains:
            residues = list(chain.get_residues())
            print(f'  Chain {chain.id}: {len(residues)} residues')
except ImportError:
    print('Biopython not available. pip install biopython')

# ── Dataset 2: CASP14 target list ───────────────────────────────────────
print('\nCASP14 targets available at:')
print('  https://predictioncenter.org/download_area/CASP14/targets/')
print('  wget https://predictioncenter.org/download_area/CASP14/targets/casp14.tgz')
print('  tar xzf casp14.tgz')
print('  Files: T1024.fasta, T1024.pdb, ... (one per target)')

# ── Dataset 3: AF3 JSON output format example ────────────────────────────
af3_output_example = {
    'atom_positions': '[[x,y,z], ...] shape: (N_atoms, 3)',
    'atom_mask': '[1,1,0,...] shape: (N_atoms,)',
    'plddt': '[85.2, 91.3, ...] per-residue confidence 0-100',
    'pae': '[[...]] shape: (N_residues, N_residues) — predicted aligned error',
    'ptm': 'float — predicted TM-score for the whole complex',
    'iptm': 'float — interface pTM for multimer predictions',
}
print('\nAF3 prediction JSON format:')
for key, desc in af3_output_example.items():
    print(f'  {key}: {desc}')
print('\nDownload from: https://alphafoldserver.com (free for academic use)')


## ✏️ Exercise: Improve the Training Loop

The current training loop trains for 300 steps on random data. Your task is to make three improvements:

1. **Gradient clipping** at `max_norm=1.0` (use `torch.nn.utils.clip_grad_norm_` — add it if not already present).

2. **Gradient-to-parameter norm ratio** every 50 steps: compute the total gradient norm divided by the total parameter norm. Log as `grad/param ratio`. This ratio should stay below ~1.0 for stable training; a spike signals an exploding gradient event.

3. **Separate loss components**: if the loop uses a combined loss `(fape_loss + distogram_loss)`, plot them on separate lines. If the loop uses a single loss, split a random fraction to simulate two components as shown in the starter code.

**Expected output:** a plot with three subplots: FAPE loss, distogram loss, and grad/param ratio over training steps.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)

# ── Minimal model (replace with the actual model from above if present) ─
class TinyPairformer(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.pair = nn.Linear(c, c)
        self.single = nn.Linear(c, c)
        self.fape_head = nn.Linear(c, 1)
        self.dist_head = nn.Linear(c, 64)

    def forward(self, x):
        h = torch.relu(self.pair(x))
        h = torch.relu(self.single(h))
        fape   = self.fape_head(h).mean()
        distogram = torch.log_softmax(self.dist_head(h), dim=-1).mean()
        return fape, -distogram  # both positive losses


model = TinyPairformer(c=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_STEPS = 300
LOG_INTERVAL = 50
MAX_GRAD_NORM = 1.0  # clipping threshold

fape_history   = []
dist_history   = []
ratio_history  = []
ratio_steps    = []

for step in range(N_STEPS):
    x = torch.randn(16, 32)  # batch_size=16, feature_dim=32
    optimizer.zero_grad()

    fape_loss, dist_loss = model(x)
    total_loss = fape_loss + dist_loss
    total_loss.backward()

    # TODO 1: Add gradient clipping here before optimizer.step()
    # torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

    optimizer.step()

    fape_history.append(fape_loss.item())
    dist_history.append(dist_loss.item())

    if step % LOG_INTERVAL == 0:
        # TODO 2: Compute and log grad/param ratio
        # grad_norm  = total L2 norm of all gradients
        # param_norm = total L2 norm of all parameters
        # ratio = grad_norm / (param_norm + 1e-8)
        # Hint: iterate model.parameters(), use p.grad and p.data
        grad_norm  = 0.0  # TODO: compute this
        param_norm = 0.0  # TODO: compute this
        ratio = grad_norm / (param_norm + 1e-8)
        ratio_history.append(ratio)
        ratio_steps.append(step)
        print(f'Step {step:3d} | FAPE={fape_loss.item():.4f} '
              f'| Dist={dist_loss.item():.4f} '
              f'| grad/param={ratio:.4f}')

# TODO 3: Plot all three metrics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(fape_history, color='steelblue', alpha=0.8)
axes[0].set_title('FAPE Loss'); axes[0].set_xlabel('Step')

axes[1].plot(dist_history, color='firebrick', alpha=0.8)
axes[1].set_title('Distogram Loss'); axes[1].set_xlabel('Step')

axes[2].plot(ratio_steps, ratio_history, 'o-', color='green', alpha=0.8)
axes[2].axhline(1.0, color='red', linestyle='--', label='ratio=1.0 (caution)')
axes[2].set_title('Grad/Param Ratio'); axes[2].set_xlabel('Step')
axes[2].legend(fontsize=8)

plt.suptitle('Training Diagnostics (Exercise)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey: grad/param ratio >> 1 indicates exploding gradients.')
print('Gradient clipping at 1.0 should keep this ratio bounded.')

# ── SOLUTION HINT (uncomment TODO 1 and 2) ────────────────────────────
# TODO 1: torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
# TODO 2:
#   grad_norm  = sum(p.grad.norm()**2 for p in model.parameters()
#                   if p.grad is not None) ** 0.5
#   param_norm = sum(p.data.norm()**2 for p in model.parameters()) ** 0.5


---
## ✅ Mastery Check — AF3 Training Loop

**Self-test without looking at the notebook.**

**Q1 (Recall):** What is FAPE loss and why is it better than RMSD for training structure prediction?
> **A:** FAPE (Frame Aligned Point Error) measures error in LOCAL coordinate frames defined by backbone rigid bodies. For each residue i, it computes the distance between predicted and true coordinates of all other residues j, expressed in residue i's coordinate frame. Unlike RMSD, FAPE is invariant to global rotation/translation and captures LOCAL structural accuracy — important because a protein can be globally correct but locally wrong (e.g., a helix pointing the right direction but with wrong internal geometry). FAPE also clamps errors at 10 Å, reducing gradient signal from catastrophically wrong regions.

**Q2 (Understand):** Why does AF3 use BF16 instead of FP16 for training?
> **A:** BF16 has the same 8-bit exponent as FP32 (range: ≈ 10^-38 to 10^38), while FP16 only has 5-bit exponent (range: ≈ 6×10^-5 to 65504). Coordinate regression and FAPE computation involve distances in Å, which can span several orders of magnitude. FP16 overflows on distances > 65504 Å and underflows on very small gradients. BF16 eliminates these numerical issues. The trade-off: BF16 has only 7 mantissa bits (vs 10 for FP16), giving lower precision but broader range — acceptable for deep learning where approximate gradients are fine.

**Q3 (Apply):** Your training loss starts at 5.0 and after 100 steps it's still 4.9. What are the first 3 things you check?
> **A:** (1) Learning rate: too low → slow convergence; too high → NaN/oscillation. Check `af3_lr_schedule` output. (2) Gradient flow: run `plot_grad_flow(model)` — if any layer shows near-zero gradient, there's a vanishing gradient problem. (3) Batch quality: print the actual FAPE and distogram loss separately — if one is stuck at exactly its initialization value, that loss component might not be connected to the model output.

**Q4 (Analyze):** Gradient accumulation is set to 4 steps with batch_size=2. What is the effective batch size? If you change gradient accumulation to 8 with batch_size=1, what changes in training dynamics?
> **A:** Effective batch size = 4 × 2 = 8. With accumulation=8, batch_size=1: same effective batch (8), but the learning rate noise from individual samples is different — single samples have higher gradient variance. This can actually help escape sharp minima (like SGD vs full-batch GD). Memory usage drops significantly (batch_size=1 vs 2). The training might be slower per step but each step requires less GPU memory.

**Q5 (Design):** You want to fine-tune this MiniAF3 on 500 real protein structures from PDB. Describe the complete setup: data processing, loss weights, early stopping criterion, and how you'd detect overfitting.
> **A:** (1) Data: parse PDB mmCIF files → extract Cα coordinates + sequence → split 400/50/50 train/val/test by BLAST identity < 30% (no sequence similarity leakage). (2) Loss: increase FAPE weight, decrease distogram weight for fine-tuning (structure matters more than distant signal). (3) Early stopping: monitor validation FAPE with patience=10; stop if val_FAPE increases for 10 consecutive epochs. (4) Overfitting detection: training FAPE << validation FAPE for 5+ epochs; or TM-score on val set starts decreasing while train TM-score improves.

**Score:**
- Q1-3: Ready to work on AF3-related tasks
- Q1-4: Strong ML engineering understanding  
- Q1-5: Interview-ready for computational biology ML roles
